# Type 1 — End-to-End Pipeline Evaluation

Chạy toàn bộ pipeline Type 1 trên bộ dữ liệu `Logic_Based_Educational_Queries.json` và đánh giá kết quả.

In [2]:
# ── 1. CẤU HÌNH ──────────────────────────────────────────────────────────────
EXACT_API_BASE_URL = "https://api.iamphuckhang.dev"  # Cloudflare tunnel trỏ tới EXACT API :8080
PREDICT_ENDPOINT   = f"{EXACT_API_BASE_URL.rstrip('/')}/predict"
DEBUG_ENDPOINT     = f"{EXACT_API_BASE_URL.rstrip('/')}/debug/predict"

# False = test đúng output chính thức của BTC; True = lấy thêm id/error/task_type để debug
USE_DEBUG_ENDPOINT = True
ACTIVE_ENDPOINT    = DEBUG_ENDPOINT if USE_DEBUG_ENDPOINT else PREDICT_ENDPOINT

REQUEST_TIMEOUT_SEC = 60.0

# None = chạy hết dataset; đặt số nguyên để giới hạn (ví dụ 50 để test nhanh)
LIMIT          = None

# Đường dẫn output
OUTPUT_PATH    = "../artifacts/predictions/type1/remote_api_eval_run.json"
REPORT_PATH    = "../artifacts/reports/type1_eval_report.json"
ERRORS_PATH    = "../artifacts/reports/type1_eval_errors.csv"

In [3]:
# ── 2. SETUP ─────────────────────────────────────────────────────────────────
import sys, json, time, csv, importlib
from pathlib import Path
from dataclasses import asdict, dataclass
from typing import Any

import requests

ROOT = Path().resolve().parent
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"root : {ROOT}")
print(f"src  : {SRC}")

# Notebook này chỉ gọi EXACT API remote; pipeline/solver/vLLM chạy trên VM.
from exact.datasets.dataset import ExactDataset
from exact.router.task_router import TaskRouter

print("✓ imports OK")

root : /home/phuckhang/MyWorkspace/Exact2026
src  : /home/phuckhang/MyWorkspace/Exact2026/src
✓ imports OK


In [5]:
# ── 3. KIỂM TRA KẾT NỐI EXACT API ───────────────────────────────────────────
def request_prediction(payload: dict[str, Any], *, endpoint: str = ACTIVE_ENDPOINT) -> dict[str, Any]:
    r = requests.post(endpoint, json=payload, timeout=REQUEST_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code} from {endpoint}: {r.text[:1000]}")
    data = r.json()
    if not isinstance(data, dict):
        raise RuntimeError(f"API response must be a JSON object, got {type(data).__name__}")
    return data

try:
    health = requests.get(f"{EXACT_API_BASE_URL.rstrip('/')}/health", timeout=15)
    print(f"health: HTTP {health.status_code} {health.text[:200]}")
    health.raise_for_status()
except Exception as e:
    print(f"✗ Không kết nối được EXACT API: {e}")

print(f"✓ endpoint: {ACTIVE_ENDPOINT}")

health: HTTP 200 {"status":"ok"}
✓ endpoint: https://api.iamphuckhang.dev/debug/predict


In [4]:
# ── 4. KIỂM TRA OUTPUT SHAPE CHÍNH THỨC ─────────────────────────────────────
print(f"API base        : {EXACT_API_BASE_URL}")
print(f"predict endpoint: {PREDICT_ENDPOINT}")
print(f"debug endpoint  : {DEBUG_ENDPOINT}")
print(f"active endpoint : {ACTIVE_ENDPOINT}")
print(f"timeout         : {REQUEST_TIMEOUT_SEC}s")

API base        : https://api.iamphuckhang.dev
predict endpoint: https://api.iamphuckhang.dev/predict
debug endpoint  : https://api.iamphuckhang.dev/debug/predict
active endpoint : https://api.iamphuckhang.dev/debug/predict
timeout         : 60.0s


In [6]:
# ── 5. LOAD DATASET ──────────────────────────────────────────────────────────
DATASET_PATH = ROOT / "src/exact/datasets/exact/Logic_Based_Educational_Queries.json"

dataset  = ExactDataset.from_file(DATASET_PATH, skip_invalid=True).filter_type1()
examples = list(dataset)
if LIMIT is not None:
    examples = examples[:LIMIT]

# Thống kê question type trong tập sẽ chạy
from collections import Counter
from exact.router.task_router import detect_question_type

router = TaskRouter()
qtype_counts: Counter = Counter()
for ex in examples:
    route = router.route(ex.request)
    qtype_counts[route.question_type.value] += 1

print(f"✓ loaded {len(examples)} examples")
print(f"  question types: {dict(qtype_counts)}")
print()
# Xem 1 mẫu
sample = examples[0]
print(f"[sample id]   {sample.request.id}")
print(f"[premises]    {sample.request.premises_nl[:2]}...")
print(f"[question]    {sample.request.question}")
print(f"[gold_answer] {sample.gold_answer}")

✓ loaded 808 examples
  question types: {'mcq': 359, 'yes_no_uncertain': 449}

[sample id]   logic_0000_00
[premises]    ['If a Python code is well-tested, then the project is optimized.', 'If a Python code does not follow PEP 8 standards, then it is not well-tested.']...
[question]    Which conclusion follows with the fewest premises?
A. If a Python project is not optimized, then it is not well-tested
B. If all Python projects are optimized, then all Python projects are well-structured
C. If a Python project is well-tested, then it must be clean and readable
D. If a Python project is not optimized, then it does not follow PEP 8 standards
[gold_answer] A


In [6]:
# Smoke test đúng layer: notebook gọi /predict, VM chạy pipeline + solver + vLLM.
smoke_payload = {
    "id": "remote_api_smoke_t1",
    "premises-NL": [
        "If a student completes assignments, the student passes.",
        "Sophia completes assignments.",
    ],
    "question": "Does Sophia pass?",
}
raw = request_prediction(smoke_payload, endpoint=PREDICT_ENDPOINT)
print(json.dumps(raw, ensure_ascii=False, indent=2))
assert "answer" in raw and "explanation" in raw
assert "id" not in raw and "error" not in raw, "Official /predict should not expose debug metadata"

{
  "answer": "Yes",
  "explanation": "Formula-Z3 judged the query as Yes. It uses entailment checks where T entails phi iff T and not(phi) is UNSAT; the translated theory status was sat.",
  "fol": "Predicates: completes_assignments/1, passes/1\nPremises:\nP1: (completes_assignments(?x) -> passes(?x))\nP2: completes_assignments(sophia)\nGoals:\nquery:passes(sophia)",
  "cot": [
    "llm_formula_translation: premises=2, goals=1, predicates=2",
    "z3_prop_theory_status: sat",
    "z3_prop_query_answer: Yes"
  ],
  "premises": [
    "P1: If a student completes assignments, the student passes.",
    "P2: Sophia completes assignments."
  ],
  "confidence": 0.78
}


In [9]:
# ── 6. CHẠY REMOTE EXACT API ────────────────────────────────────────────────

predictions: list[dict[str, Any]] = []
counters = {"correct": 0, "wrong": 0, "error": 0}
total = len(examples)
t_start = time.time()

for idx, example in enumerate(examples, start=1):
    route = router.route(example.request)
    t0 = time.time()
    try:
        payload = example.request.model_dump(mode="json", by_alias=True, exclude_none=True)
        response = request_prediction(payload)
        pred_error = response.get("error")
    except Exception as exc:
        pred_error = f"{type(exc).__name__}: {exc}"
        response = {
            "answer": "",
            "explanation": f"pipeline error: {pred_error}",
            "fol": None,
            "cot": [],
            "premises": [],
            "confidence": 0.0,
        }

    elapsed = time.time() - t0

    # Chấm điểm ngay
    ans      = str(response.get("answer") or "").strip().lower()
    gold     = (example.gold_answer or "").strip().lower()
    if (pred_error or str(response.get("explanation") or "").startswith("Prediction failed")) and not ans:
        status = "error"
    elif ans == gold:
        status = "correct"
    else:
        status = "wrong"
    counters[status] += 1

    # Lưu prediction dạng phẳng, không lặp field trong "official"
    prediction = {
        "id"           : response.get("id") or example.request.id,
        "task_type"    : response.get("task_type") or "type1_logic",
        "question_type": response.get("question_type") or route.question_type.value,
        "answer"       : response.get("answer", ""),
        "gold_answer"  : example.gold_answer,
        "explanation"  : response.get("explanation"),
        "fol"          : response.get("fol"),
        "cot"          : response.get("cot"),
        "premises"     : response.get("premises"),
        "confidence"   : response.get("confidence"),
        "error"        : pred_error,
        "route_reason" : route.reason,
        "_elapsed_s"   : round(elapsed, 2),
        "_status"      : status,
    }
    predictions.append(prediction)

    # In progress mỗi 10 item hoặc item cuối
    if idx % 10 == 0 or idx == total:
        scored   = counters["correct"] + counters["wrong"] + counters["error"]
        accuracy = counters["correct"] / scored if scored else 0
        elapsed_total = time.time() - t_start
        eta = (elapsed_total / idx) * (total - idx)
        print(
            f"[{idx:4d}/{total}] "
            f"acc={accuracy:.3f} "
            f"✓{counters['correct']} ✗{counters['wrong']} ⚡{counters['error']} "
            f"| elapsed={elapsed_total:.0f}s ETA={eta:.0f}s"
        )

print(f"✓ done — {total} predictions in {time.time()-t_start:.0f}s")

[  10/808] acc=0.600 ✓6 ✗3 ⚡1 | elapsed=129s ETA=10288s
[  20/808] acc=0.600 ✓12 ✗7 ⚡1 | elapsed=348s ETA=13699s
[  30/808] acc=0.533 ✓16 ✗12 ⚡2 | elapsed=518s ETA=13446s
[  40/808] acc=0.575 ✓23 ✗15 ⚡2 | elapsed=613s ETA=11767s
[  50/808] acc=0.480 ✓24 ✗16 ⚡10 | elapsed=1053s ETA=15962s
[  60/808] acc=0.450 ✓27 ✗16 ⚡17 | elapsed=1424s ETA=17758s


KeyboardInterrupt: 

In [8]:
# ── 7. LƯU PREDICTIONS ───────────────────────────────────────────────────────
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

output = {
    "api_base_url": EXACT_API_BASE_URL,
    "endpoint"    : ACTIVE_ENDPOINT,
    "use_debug_endpoint": USE_DEBUG_ENDPOINT,
    "limit"      : LIMIT,
    "count"      : len(predictions),
    "format"     : "exact_predictions",
    "predictions": predictions,
}
output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ saved {len(predictions)} predictions → {output_path}")

✓ saved 190 predictions → ../artifacts/predictions/type1/remote_api_eval_run.json


In [ ]:
# ── 8. ĐÁNH GIÁ ──────────────────────────────────────────────────────────────
@dataclass(frozen=True)
class EvalRow:
    id: str | None
    answer: str
    gold_answer: str | None
    question_type: str | None
    status: str
    error: str | None

def _safe_ratio(n: int, d: int) -> float:
    return n / d if d else 0.0

def evaluate_all(preds: list[dict]) -> tuple[list[EvalRow], dict]:
    rows: list[EvalRow] = []
    for p in preds:
        ans      = str(p.get("answer") or "").strip()
        gold     = str(p.get("gold_answer") or "").strip()
        err      = str(p.get("error") or "").strip() or None
        qtype    = str(p.get("question_type") or "unknown")
        if not gold:
            status = "missing_gold"
        elif ans.lower() == gold.lower():
            status = "correct"
        elif err and not ans:
            status = "pipeline_error"
        else:
            status = "wrong"
        rows.append(EvalRow(id=p.get("id"), answer=ans, gold_answer=gold,
                            question_type=qtype, status=status, error=err))

    total         = len(rows)
    missing_gold  = sum(r.status == "missing_gold"  for r in rows)
    scored        = total - missing_gold
    correct       = sum(r.status == "correct"        for r in rows)
    pipe_errors   = sum(r.status == "pipeline_error" for r in rows)
    wrong         = scored - correct

    # Per question type
    grouped: dict[str, list[EvalRow]] = {}
    for r in rows:
        if r.status != "missing_gold":
            grouped.setdefault(r.question_type or "unknown", []).append(r)
    by_qtype = {
        qt: {
            "total"          : len(g),
            "correct"        : sum(r.status == "correct" for r in g),
            "accuracy"       : _safe_ratio(sum(r.status == "correct" for r in g), len(g)),
            "pipeline_errors": sum(r.status == "pipeline_error" for r in g),
        }
        for qt, g in sorted(grouped.items())
    }

    summary = {
        "total"          : total,
        "scored_total"   : scored,
        "correct"        : correct,
        "wrong"          : wrong,
        "missing_gold"   : missing_gold,
        "accuracy"       : _safe_ratio(correct, scored),
        "pipeline_errors": pipe_errors,
        "by_question_type": by_qtype,
    }
    return rows, summary

rows, summary = evaluate_all(predictions)

print("══ SUMMARY ══════════════════════════════")
print(f"  total          : {summary['total']}")
print(f"  scored_total   : {summary['scored_total']}")
print(f"  correct        : {summary['correct']}")
print(f"  wrong          : {summary['wrong']}")
print(f"  pipeline_errors: {summary['pipeline_errors']}")
print(f"  ACCURACY       : {summary['accuracy']:.4f}  ({summary['accuracy']*100:.2f}%)")
print()
print("══ BY QUESTION TYPE ══════════════════════")
for qt, stats in summary["by_question_type"].items():
    print(f"  {qt:<25} acc={stats['accuracy']:.4f}  ({stats['correct']}/{stats['total']})  errors={stats['pipeline_errors']}")

In [ ]:
# ── 9. LƯU REPORT VÀ ERRORS CSV ──────────────────────────────────────────────
report_path = Path(REPORT_PATH)
errors_path = Path(ERRORS_PATH)
report_path.parent.mkdir(parents=True, exist_ok=True)
errors_path.parent.mkdir(parents=True, exist_ok=True)

report = {
    "source"   : str(output_path),
    "api_base_url": EXACT_API_BASE_URL,
    "endpoint"    : ACTIVE_ENDPOINT,
    "count"    : len(rows),
    "summary"  : summary,
    "rows"     : [asdict(r) for r in rows],
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ report  → {report_path}")

non_correct = [r for r in rows if r.status != "correct"]
if non_correct:
    with errors_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(non_correct[0]).keys()))
        writer.writeheader()
        writer.writerows(asdict(r) for r in non_correct)
    print(f"✓ errors  → {errors_path}  ({len(non_correct)} rows)")
else:
    print("✓ no errors to write")

In [ ]:
# ── 10. PHÂN TÍCH LỖI (hiển thị 20 case sai/lỗi đầu tiên) ───────────────────
error_cases = [r for r in rows if r.status != "correct" and r.status != "missing_gold"]
print(f"Non-correct cases: {len(error_cases)}")
print()

for i, row in enumerate(error_cases[:20], 1):
    print(f"[{i:2d}] id={row.id}  type={row.question_type}  status={row.status}")
    print(f"      pred={row.answer!r}  gold={row.gold_answer!r}")
    if row.error:
        print(f"      error={row.error[:120]}")
    print()

In [ ]:
# ── 11. PHÂN PHỐI CONFIDENCE ─────────────────────────────────────────────────
import statistics

conf_correct = [p["confidence"] for p in predictions if p.get("confidence") and p["_status"] == "correct"]
conf_wrong   = [p["confidence"] for p in predictions if p.get("confidence") and p["_status"] == "wrong"]

def _stats(vals: list[float], label: str) -> None:
    if not vals:
        print(f"  {label}: no data")
        return
    print(f"  {label} (n={len(vals)}):  mean={statistics.mean(vals):.3f}  "
          f"median={statistics.median(vals):.3f}  "
          f"min={min(vals):.3f}  max={max(vals):.3f}")

print("Confidence distribution:")
_stats(conf_correct, "correct")
_stats(conf_wrong,   "wrong  ")